# MacroABM-CA quick start

This notebook runs the Canadian provincial model **on its own**: no CIMS, COPPER or
LabourABM. It covers four things:

1. run the baseline, 2022 to 2030;
2. look at the results;
3. define a scenario and run it;
4. compare it with the baseline.

**Before you start:** follow `docs/canada/canada_beta_model_onboarding_guide.md`
(install, plus the data file this notebook needs).

**Time:** about 1 minute to load the data, then about 3 minutes per run to 2030
(two runs). **Memory:** about 4 GB.
The first run is a little slower while the model's compiled functions are built.

## 1. Settings

The only things you should need to change are in this cell.

In [ ]:
from pathlib import Path

PKL_PATH = Path("dev/pkl_files/io2022_10prov_2022_v2_pumf_built-2026-08-18.pkl")  # the starting data
END_YEAR = 2030               # the model starts in 2022Q1 and steps one quarter at a time
OUTPUT_DIR = Path("dev/quickstart_output")  # where results are saved (dev/ is not tracked by git)

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # library deprecation notices, not model problems

import matplotlib.pyplot as plt
import pandas as pd

from scenarios.macro_only import (
    BASELINE, load_data, build, run, results, run_scenario, annual, industry_output, save,
)

pd.options.display.float_format = "{:,.2f}".format

## 2. Load the data

The pickle holds the model's starting point for 2022: the provincial input-output tables,
the firms, households, banks and governments of each of the 10 provinces, and the rest of
the world. Loading it builds nothing new; it is read once and reused by every run below.

In [ ]:
data = load_data(PKL_PATH)
print(len(data.industries), "industries:", ", ".join(data.industries[:8]), "...")

## 3. Run the baseline

`build` creates the simulation with the baseline settings, `run` steps it forward.
The settings are in `BASELINE` (printed below); they are the macro settings the
CIMS-COPPER linkage uses for its EPM baseline, without the parts that need the
linkage's energy inputs.

In [ ]:
for key, value in BASELINE.items():
    print(f"{key:40s} {value}")

In [ ]:
baseline = build(data, end_year=END_YEAR)
run(baseline)
base = results(baseline)

## 4. Look at the results

`results` gives one row per province and quarter. Money is in nominal Canadian dollars
per quarter; `Real GDP` is GDP deflated by the province's CPI.

In [ ]:
print(len(base.columns), "columns:", ", ".join(base.columns))
base.head()

`annual` turns it into yearly numbers, for each province or for Canada (`national=True`).
Flows are summed over the year; rates and prices are averaged.

In [ ]:
canada = annual(base, ["Real GDP", "GDP Output", "Unemployment Rate", "CPI"], national=True)
canada["Real GDP growth %"] = canada["Real GDP"].pct_change() * 100
canada["Inflation %"] = canada["CPI"].pct_change() * 100
canada[["Real GDP", "GDP Output"]] /= 1e9   # $ billions
canada["Unemployment Rate"] *= 100
canada.round(2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
canada["Real GDP"].plot(ax=ax[0], title="Real GDP, Canada ($ billions, 2022 prices)")
canada["Unemployment Rate"].plot(ax=ax[1], title="Unemployment rate (%)")
canada["Inflation %"].plot(ax=ax[2], title="CPI inflation (%)")
plt.tight_layout()

By province: real GDP as an index (2022 = 1).

In [ ]:
prov = annual(base, ["Real GDP"])["Real GDP"].unstack("province")
(prov / prov.iloc[0]).plot(figsize=(9, 5), title="Real GDP by province (2022 = 1)")
plt.ylabel("index")
plt.show()

By industry: which industries grow most (real production, all provinces).
Industry codes follow the input-output tables (for example `F` construction,
`D` electricity, `B06` oil and gas extraction).

In [ ]:
by_industry = industry_output(baseline)
first, last = by_industry.iloc[:4].sum(), by_industry.iloc[-4:].sum()   # first and last year
change = ((last / first - 1) * 100).sort_values()
change.plot.barh(figsize=(8, 10), title=f"Real production, {END_YEAR} vs 2022 (%)")
plt.show()

Save the results. The file has the same layout as the `simulation_shallow.h5` the
linkage writes, so the same scripts read both.

In [ ]:
print(save(baseline, OUTPUT_DIR / "baseline"))

A run holds a few GB of memory. We have its results in `base`, so free it before running
more.

In [ ]:
del baseline

## 5. Define and run a scenario

A scenario is the baseline with something changed. The simplest change is a setting:
pass it by name. Any key of `BASELINE` works, and so does any other argument of
`build_simulation` in `scenarios/run_cims_linkage.py`.

Here households want 2% more goods and services per year instead of 1%.
`run_scenario` builds and runs like `build` + `run` above, but returns only the results
table and lets go of the simulation, so memory stays low.

In [ ]:
scenario = run_scenario(data, end_year=END_YEAR, household_demand_growth=0.02)

## 6. Compare with the baseline

Both runs use the same random seed (`seed` in `BASELINE`), so the difference comes from
the scenario, not from luck. The model is stochastic, though: before reporting a small
difference, rerun the pair with a few other seeds (`run_scenario(data, seed=1)`, ...)
and check the sign holds.

In [ ]:
columns = ["Real GDP", "Household Consumption", "Imports", "CPI"]
s = annual(scenario, columns, national=True)
b = annual(base, columns, national=True)
compare = (s / b - 1) * 100
compare.columns = [f"{c} %" for c in columns]
compare["Unemployment rate, pp"] = (
    annual(scenario, ["Unemployment Rate"], national=True)["Unemployment Rate"]
    - annual(base, ["Unemployment Rate"], national=True)["Unemployment Rate"]
) * 100
compare.round(2)

In [ ]:
compare.plot(figsize=(9, 5), title="Scenario vs baseline, Canada")
plt.axhline(0, color="grey", lw=0.8)
plt.ylabel("% (unemployment: percentage points)")
plt.show()

## 7. Next steps

- **Other settings to try:** `tfp_base_growth_rate` (productivity growth per quarter),
  `wage_cpi_indexation` (how much wages follow prices), `labour_force_growth` (per
  province), `seed`. Each is explained where it is used, in `build_simulation`
  (`scenarios/run_cims_linkage.py`).
- **Changes during the run (hooks):** to change the economy from a given year (a tax,
  a purchase programme), write a hook; see the guide, section 6, and the example
  `extra_government_purchases` in `scenarios/macro_only.py`.
- **Energy and climate policy:** carbon pricing, ITCs, energy use and electricity supply
  come from the CIMS-COPPER linkage (the `M3-linkages` repository), not from this
  notebook. See the guide, section "Running with the linkage".